In [2]:
%pip install pymupdf pytesseract pandas pillow

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path

desktop = Path.home() / "Desktop"

matches = list(desktop.rglob("tesseract.exe"))

print(matches)

[WindowsPath('C:/Users/USER/Desktop/tesseract.exe')]


In [5]:
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\USER\Desktop\tesseract.exe"
)

print("Tesseract version:", pytesseract.get_tesseract_version())

Tesseract version: 5.5.3.20260724


In [7]:
page_3_text = ocr_page(pdf_path, 3)
print(page_3_text)

OUTPUT BASED ANNUAL BUDGET Page 3
HEA 9811 CHIRUNDU TOWN COUNCIL
D
a
001 Residential 202,069 202,069 202,069
004 Hospitality 294,521 294,521 294,521
a
002 Survey fees 28,600 30,030 31,532
064 Hire of Transport and Equipment 250,000 262,500 275,625
067 Ablution Fee 80,000 84,000 88,200
082 Telecommunication site rentals 35,000 36,750 38,588
| [ueenses |
001 Occupancy licence 14,500 15,225 15,986
003 Firearm and ammunition licence 25,980 27,279 28,643



In [10]:
def ocr_page(pdf_path, page_number):
    document = pymupdf.open(pdf_path)

    try:
        page = document[page_number - 1]

        pix = page.get_pixmap(
            matrix=pymupdf.Matrix(300 / 72, 300 / 72),
            alpha=False
        )

        image = Image.frombytes(
            "RGB",
            [pix.width, pix.height],
            pix.samples
        )

        image = ImageOps.grayscale(image)
        image = image.point(lambda pixel: 0 if pixel < 180 else 255)

        return pytesseract.image_to_string(
            image,
            config="--oem 3 --psm 4"
        )

    finally:
        document.close()

In [11]:
page_3_text = ocr_page(pdf_path, 3)
print(page_3_text)

OUTPUT BASED ANNUAL BUDGET Page 3

HEA 9811 CHIRUNDU TOWN COUNCIL
D

01 Local taxes/rates
001 Residential 202,069 202,069 202,069
002 Commercial 253,425 253,425 253,425
004 Hospitality 294,521 294,521 294,521
001 Personal levy 45,000 47,250 49,613

02 Fees and Charges

002 Survey fees 28,600 30,030 31,532
003 Building inspection-fees 20,000 21,000 22,050
004 Plan scrutiny fee 50,041 52,543 55,171
005 Change of premise use 27,800 29,190 30,650
007 Rentals/lease of Council’s properties 9,100 9,555 10,033
011 Search fees 500 525 551
012 Notice board advert fees 3,000 3,150 3,308
013 Market fees 50,206 52,716 55,352
014 Parking fees 28,078,895 28,108,500 29,513,925
016 Loading fees (buses, trucks, trains, taxies etc.) 31,500 33,075 34,729
020 Hire of halls 10,000 10,500 11,025
033 Refuse disposal 200,000 210,000 220,500
045 Notice of marriage fees 20,000 21,000 22,050
047 Registration of clubs and societies 11,600 12,180 12,789
059 Land Record 10,000 10,500 11,025
063 Billboards and banner

In [12]:
document = pymupdf.open(pdf_path)
total_pages = len(document)
document.close()

raw_ocr_rows = []

for page_number in range(1, total_pages + 1):
    print(f"Reading page {page_number} of {total_pages}...")

    raw_ocr_rows.append({
        "year": YEAR,
        "page": page_number,
        "ocr_text": ocr_page(pdf_path, page_number),
        "source_file": pdf_path.name
    })

raw_ocr = pd.DataFrame(raw_ocr_rows)

raw_file = output_folder / "chirundu_obb_2025_raw_ocr.csv"

raw_ocr.to_csv(
    raw_file,
    sep="|",
    index=False,
    encoding="utf-8-sig"
)

print("OCR complete.")
print("Pages processed:", len(raw_ocr))
print("Raw OCR file:", raw_file)

Reading page 1 of 58...
Reading page 2 of 58...
Reading page 3 of 58...
Reading page 4 of 58...
Reading page 5 of 58...
Reading page 6 of 58...
Reading page 7 of 58...
Reading page 8 of 58...
Reading page 9 of 58...
Reading page 10 of 58...
Reading page 11 of 58...
Reading page 12 of 58...
Reading page 13 of 58...
Reading page 14 of 58...
Reading page 15 of 58...
Reading page 16 of 58...
Reading page 17 of 58...
Reading page 18 of 58...
Reading page 19 of 58...
Reading page 20 of 58...
Reading page 21 of 58...
Reading page 22 of 58...
Reading page 23 of 58...
Reading page 24 of 58...
Reading page 25 of 58...
Reading page 26 of 58...
Reading page 27 of 58...
Reading page 28 of 58...
Reading page 29 of 58...
Reading page 30 of 58...
Reading page 31 of 58...
Reading page 32 of 58...
Reading page 33 of 58...
Reading page 34 of 58...
Reading page 35 of 58...
Reading page 36 of 58...
Reading page 37 of 58...
Reading page 38 of 58...
Reading page 39 of 58...
Reading page 40 of 58...
Reading p

In [13]:
extracted_columns = [
    "year",
    "category",
    "programme_or_output",
    "page",
    "budget_amount",
    "currency",
    "notes",
    "source_file",
    "target_value",
    "target_unit"
]

extracted_rows = []

financial_rows = []

for page_number in [3, 4]:
    page_text = raw_ocr.loc[
        raw_ocr["page"] == page_number,
        "ocr_text"
    ].iloc[0]

    current_group = None

    for line in page_text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()

        # Example: 01 Local taxes/rates
        group_match = re.match(r"^(0[1-9])\s+([A-Za-z].+)$", line)

        if group_match:
            current_group = group_match.group(2)
            continue

        # Example: 001 Residential 202,069 202,069 202,069
        item_match = re.match(
            r"^(\d{3})\s+(.+?)\s+(\d[\d,]*)\s+(\d[\d,]*)\s+(\d[\d,]*)$",
            line
        )

        if not item_match or current_group is None:
            continue

        revenue_code = item_match.group(1)
        item_name = item_match.group(2).strip()

        # First figure is the 2025 budget.
        budget_2025 = int(item_match.group(3).replace(",", ""))

        if "Constituency Development Fund" in item_name:
            category = "CDF"

        elif "Local Government Equalisation Fund" in item_name:
            category = "LGEF"

        elif current_group in [
            "Local taxes/rates",
            "Fees and Charges",
            "Licenses",
            "Levies",
            "Permits",
            "Charges",
            "Other Incomes"
        ]:
            category = "Local Revenue"

        else:
            category = "Revenue"

        financial_rows.append({
            "year": YEAR,
            "category": category,
            "programme_or_output": item_name,
            "page": page_number,
            "budget_amount": budget_2025,
            "currency": "ZMW",
            "notes": f"Revenue code {revenue_code}; group: {current_group}.",
            "source_file": pdf_path.name,
            "target_value": None,
            "target_unit": None
        })

financial_df = pd.DataFrame(financial_rows, columns=extracted_columns)

display(financial_df)
print("Financial rows extracted:", len(financial_df))

,year,category,programme_or_output,page,budget_amount,currency,notes,source_file,target_value,target_unit
0,2025,Local Revenue,Residential,3,202069,ZMW,Revenue code 001; group: Local taxes/rates.,chirundu_obb_2025.pdf.pdf,None,None
1,2025,Local Revenue,Commercial,3,253425,ZMW,Revenue code 002; group: Local taxes/rates.,chirundu_obb_2025.pdf.pdf,None,None
2,2025,Local Revenue,Hospitality,3,294521,ZMW,Revenue code 004; group: Local taxes/rates.,chirundu_obb_2025.pdf.pdf,None,None
3,2025,Local Revenue,Personal levy,3,45000,ZMW,Revenue code 001; group: Local taxes/rates.,chirundu_obb_2025.pdf.pdf,None,None
4,2025,Local Revenue,Survey fees,3,28600,ZMW,Revenue code 002; group: Fees and Charges.,chirundu_obb_2025.pdf.pdf,None,None
5,2025,Local Revenue,Building inspection-fees,3,20000,ZMW,Revenue code 003; group: Fees and Charges.,chirundu_obb_2025.pdf.pdf,None,None
6,2025,Local Revenue,Plan scrutiny fee,3,50041,ZMW,Revenue code 004; group: Fees and Charges.,chirundu_obb_2025.pdf.pdf,None,None
7,2025,Local Revenue,Change of premise use,3,27800,ZMW,Revenue code 005; group: Fees and Charges.,chirundu_obb_2025.pdf.pdf,None,None
8,2025,Local Revenue,Rentals/lease of Council’s properties,3,9100,ZMW,Revenue code 007; group: Fees and Charges.,chirundu_obb_2025.pdf.pdf,None,None
9,2025,Local Revenue,Search fees,3,500,ZMW,Revenue code 011; group: Fees and Charges.,chirundu_obb_2025.pdf.pdf,None,None


Financial rows extracted: 56


In [14]:
missing_other_income = {
    "year": 2025,
    "category": "Local Revenue",
    "programme_or_output": "Surplus/Deficit from Commercial Ventures",
    "page": 4,
    "budget_amount": 1000000,
    "currency": "ZMW",
    "notes": "Revenue code 002; group: Other Incomes. Added after OCR review.",
    "source_file": pdf_path.name,
    "target_value": None,
    "target_unit": None
}

financial_df = pd.concat(
    [financial_df, pd.DataFrame([missing_other_income])],
    ignore_index=True
)

print("Financial rows after correction:", len(financial_df))

Financial rows after correction: 57


In [15]:
extracted_rows = financial_df.to_dict("records")

In [17]:
def show_page(page_number):
    page_text = raw_ocr.loc[
        raw_ocr["page"] == page_number,
        "ocr_text"
    ].iloc[0]

    print(page_text)

In [18]:
show_page(7)

OUTPUT BASED ANNUAL BUDGET Page 7

HEA 9811 CHIRUNDU TOWN COUNCIL
D

Table:2. Budget Allocation by Programme

2023 2024 2025
Code Programme Approved Approved Budget

Budget(K) Budget(K) | Estimates(K)

1 Constituency Development (0) 30,635,642 36,058,150
2 Local Governance (0) 1,377,628 3,548,140
3 Integrated Development Planning (0) 3,095,245 5,329,132
4 Economic and Business Development (0) 676,430 1,007,948
5 Public Health and Environmental Protection (0) 3,807,485 4,038,834
6 Housing and Community Amenities (0) 13,182,722 10,232,353
7 Recreation Culture and Religion (0) 642,134 736,244
8 Education and Skills Development (0) 228,100 19,241
10 Public Order and Safety (0) 677,220 5,253,277
11 Management and Support Services (0) 10,951,590 13,868,479
12 Resource Mobilisation and Management (0) 9,150,433 6,883,118
13 District Health servcies (0) 1,457,388 1,467,387
15 Transport Services (0) 3,742,847 3,252,077
16 Agricultural Services (0) (0) 481,034
17 Fisheries and Livestock (0) (0) 7

In [19]:
programme_rows = []

page_7_text = raw_ocr.loc[
    raw_ocr["page"] == 7,
    "ocr_text"
].iloc[0]

for line in page_7_text.splitlines():
    line = re.sub(r"\s+", " ", line).strip()

    programme_match = re.match(
        r"^(\d{1,2})\s+(.+?)\s+"
        r"(?:\(0\)|-|\d[\d,]*)\s+"
        r"(?:\(0\)|-|\d[\d,]*)\s+"
        r"(\d[\d,]*)$",
        line
    )

    if programme_match:
        programme_code = programme_match.group(1)
        programme_name = programme_match.group(2).strip()
        budget_2025 = int(programme_match.group(3).replace(",", ""))

        programme_rows.append({
            "year": YEAR,
            "category": "Programme",
            "programme_or_output": programme_name,
            "page": 7,
            "budget_amount": budget_2025,
            "currency": "ZMW",
            "notes": f"Programme code {programme_code}; 2025 budget allocation.",
            "source_file": pdf_path.name,
            "target_value": None,
            "target_unit": None
        })

programmes_df = pd.DataFrame(
    programme_rows,
    columns=extracted_columns
)

# Correct the OCR spelling before adding it to the final dataset.
programmes_df["programme_or_output"] = programmes_df[
    "programme_or_output"
].replace(
    "District Health servcies",
    "District Health Services"
)

display(programmes_df)
print("Programme rows found:", len(programmes_df))

,year,category,programme_or_output,page,budget_amount,currency,notes,source_file,target_value,target_unit
0,2025,Programme,Constituency Development,7,36058150,ZMW,Programme code 1; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None
1,2025,Programme,Local Governance,7,3548140,ZMW,Programme code 2; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None
2,2025,Programme,Integrated Development Planning,7,5329132,ZMW,Programme code 3; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None
3,2025,Programme,Economic and Business Development,7,1007948,ZMW,Programme code 4; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None
4,2025,Programme,Public Health and Environmental Protection,7,4038834,ZMW,Programme code 5; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None
5,2025,Programme,Housing and Community Amenities,7,10232353,ZMW,Programme code 6; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None
6,2025,Programme,Recreation Culture and Religion,7,736244,ZMW,Programme code 7; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None
7,2025,Programme,Education and Skills Development,7,19241,ZMW,Programme code 8; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None
8,2025,Programme,Public Order and Safety,7,5253277,ZMW,Programme code 10; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None
9,2025,Programme,Management and Support Services,7,13868479,ZMW,Programme code 11; 2025 budget allocation.,chirundu_obb_2025.pdf.pdf,None,None


Programme rows found: 16


In [20]:
extracted_rows.extend(programmes_df.to_dict("records"))

print("Total rows so far:", len(extracted_rows))

Total rows so far: 73


In [21]:
output_rows = []

for _, row in raw_ocr.iterrows():
    page_text = row["ocr_text"]

    programme_match = re.search(
        r"Programme\s*:?\s*0*(\d+)\s*:?\s*([A-Za-z][^\n]+)",
        page_text,
        flags=re.IGNORECASE
    )

    output_section = re.search(
        r"Table\s*6\s*:\s*Programme Outputs(.*?)(?:Executive Authority|Controlling Officer)",
        page_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not programme_match or not output_section:
        continue

    programme_number = programme_match.group(1)
    programme_name = programme_match.group(2).strip()
    current_key_output = None

    for line in output_section.group(1).splitlines():
        line = re.sub(r"\s+", " ", line).strip()

        if not line or "Key Output" in line or "Target" in line or "Actual" in line:
            continue

        indicator_match = re.match(r"^(\d{2})\.?\s+(.+)$", line)

        if indicator_match:
            indicator_number = indicator_match.group(1)
            indicator_and_values = indicator_match.group(2)

            values = re.findall(
                r"(?<![A-Za-z])(?:\(?\d[\d,]*\)?|-)(?![A-Za-z])",
                indicator_and_values
            )

            target_raw = values[-1] if values else None

            indicator = re.sub(
                r"\s+(?:\(?\d[\d,]*\)?|-)"
                r"(?:\s+(?:\(?\d[\d,]*\)?|-))*\s*$",
                "",
                indicator_and_values
            ).strip()

            if "percentage" in indicator.lower():
                target_unit = "percent"
            elif "number" in indicator.lower():
                target_unit = "count"
            else:
                target_unit = "value"

            if target_raw in [None, "-"]:
                target_value = None
            else:
                target_value = int(
                    target_raw
                    .replace("(", "")
                    .replace(")", "")
                    .replace(",", "")
                )

            output_rows.append({
                "year": YEAR,
                "category": "Output",
                "programme_or_output": indicator,
                "page": row["page"],
                "budget_amount": None,
                "currency": "ZMW",
                "notes": (
                    f"Programme {programme_number}: {programme_name}; "
                    f"Key output: {current_key_output}; "
                    f"Indicator {indicator_number}."
                ),
                "source_file": pdf_path.name,
                "target_value": target_value,
                "target_unit": target_unit
            })

outputs_df = pd.DataFrame(
    output_rows,
    columns=extracted_columns
)

display(outputs_df.head(10))
print("Output rows found:", len(outputs_df))

,year,category,programme_or_output,page,budget_amount,currency,notes,source_file,target_value,target_unit
0,2025,Output,Percentage of Community Projects implemented,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
1,2025,Output,Percentage of approved grant applicants empowered,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
2,2025,Output,Percentage of approved loan applicants empowered,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
3,2025,Output,Number of monitoring and evaluation activities...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,8.0,count
4,2025,Output,Number of project appraisals carried out,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,2.0,count
5,2025,Output,Percentage approved Skills bursaries applicant...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
6,2025,Output,Percentage approved Secondary bursaries applic...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
7,2025,Output,Number of Ordinary Council Meetings Held,16,None,ZMW,Programme 2: Local Governance; Key output: Non...,chirundu_obb_2025.pdf.pdf,4.0,count
8,2025,Output,Number of Committee Meetings Held,16,None,ZMW,Programme 2: Local Governance; Key output: Non...,chirundu_obb_2025.pdf.pdf,21.0,count
9,2025,Output,Number of Special Full Council Meetings Held,16,None,ZMW,Programme 2: Local Governance; Key output: Non...,chirundu_obb_2025.pdf.pdf,2.0,count


Output rows found: 139


In [22]:
def ocr_page(pdf_path, page_number, psm=4):
    document = pymupdf.open(pdf_path)

    try:
        page = document[page_number - 1]

        pix = page.get_pixmap(
            matrix=pymupdf.Matrix(300 / 72, 300 / 72),
            alpha=False
        )

        image = Image.frombytes(
            "RGB",
            [pix.width, pix.height],
            pix.samples
        )

        image = ImageOps.grayscale(image)
        image = image.point(lambda pixel: 0 if pixel < 180 else 255)

        return pytesseract.image_to_string(
            image,
            config=f"--oem 3 --psm {psm}"
        )

    finally:
        document.close()

In [23]:
for page_number in range(12, total_pages + 1):
    print(f"Re-reading output page {page_number} of {total_pages}...")

    raw_ocr.loc[
        raw_ocr["page"] == page_number,
        "ocr_text"
    ] = ocr_page(pdf_path, page_number, psm=6)

raw_ocr.to_csv(
    output_folder / "chirundu_obb_2025_raw_ocr.csv",
    sep="|",
    index=False,
    encoding="utf-8-sig"
)

print("Output pages re-OCR completed.")

Re-reading output page 12 of 58...
Re-reading output page 13 of 58...
Re-reading output page 14 of 58...
Re-reading output page 15 of 58...
Re-reading output page 16 of 58...
Re-reading output page 17 of 58...
Re-reading output page 18 of 58...
Re-reading output page 19 of 58...
Re-reading output page 20 of 58...
Re-reading output page 21 of 58...
Re-reading output page 22 of 58...
Re-reading output page 23 of 58...
Re-reading output page 24 of 58...
Re-reading output page 25 of 58...
Re-reading output page 26 of 58...
Re-reading output page 27 of 58...
Re-reading output page 28 of 58...
Re-reading output page 29 of 58...
Re-reading output page 30 of 58...
Re-reading output page 31 of 58...
Re-reading output page 32 of 58...
Re-reading output page 33 of 58...
Re-reading output page 34 of 58...
Re-reading output page 35 of 58...
Re-reading output page 36 of 58...
Re-reading output page 37 of 58...
Re-reading output page 38 of 58...
Re-reading output page 39 of 58...
Re-reading output pa

In [24]:
output_rows = []

In [25]:
print("Output rows found:", len(outputs_df))
display(outputs_df.head(10))

Output rows found: 139


,year,category,programme_or_output,page,budget_amount,currency,notes,source_file,target_value,target_unit
0,2025,Output,Percentage of Community Projects implemented,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
1,2025,Output,Percentage of approved grant applicants empowered,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
2,2025,Output,Percentage of approved loan applicants empowered,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
3,2025,Output,Number of monitoring and evaluation activities...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,8.0,count
4,2025,Output,Number of project appraisals carried out,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,2.0,count
5,2025,Output,Percentage approved Skills bursaries applicant...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
6,2025,Output,Percentage approved Secondary bursaries applic...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2025.pdf.pdf,100.0,percent
7,2025,Output,Number of Ordinary Council Meetings Held,16,None,ZMW,Programme 2: Local Governance; Key output: Non...,chirundu_obb_2025.pdf.pdf,4.0,count
8,2025,Output,Number of Committee Meetings Held,16,None,ZMW,Programme 2: Local Governance; Key output: Non...,chirundu_obb_2025.pdf.pdf,21.0,count
9,2025,Output,Number of Special Full Council Meetings Held,16,None,ZMW,Programme 2: Local Governance; Key output: Non...,chirundu_obb_2025.pdf.pdf,2.0,count


In [26]:
outputs_df["programme_code"] = outputs_df["notes"].str.extract(
    r"Programme\s+(\d+)"
)[0]

programme_output_counts = (
    outputs_df
    .groupby("programme_code")
    .size()
    .reset_index(name="output_indicator_count")
)

display(programme_output_counts)

expected_programmes = {
    "1", "2", "3", "4", "5", "6", "7", "8",
    "10", "11", "12", "13", "15", "16", "17", "18"
}

found_programmes = set(outputs_df["programme_code"].dropna())

print("Missing programmes:", sorted(expected_programmes - found_programmes))
print("Found programmes:", sorted(found_programmes, key=int))

,programme_code,output_indicator_count
0,1,7
1,10,4
2,11,4
3,12,1
4,13,13
5,15,6
6,17,27
7,18,17
8,2,6
9,3,18


Missing programmes: ['16']
Found programmes: ['1', '2', '3', '4', '5', '6', '7', '8', '10', '11', '12', '13', '15', '17', '18']


In [27]:
agriculture_pages = raw_ocr[
    raw_ocr["ocr_text"].str.contains(
        "Agricultural Services|Agricultural Crop|Agribusiness",
        case=False,
        na=False
    )
][["page", "ocr_text"]]

display(agriculture_pages)

,page,ocr_text
6,7,OUTPUT BASED ANNUAL BUDGET Page 7\n\nHEA 9811 ...
8,9,OUTPUT BASED ANNUAL BUDGET Page 9\n\nHEA 9811 ...
10,11,OUTPUT BASED ANNUAL BUDGET Page 11\n\nHEA 9811...
39,40,Page 40 OUTPUT BASED ANNUAL BUDGET\nHEA 9811 C...
40,41,OUTPUT BASED ANNUAL BUDGET Page 41\nHEA 9811 C...
41,42,Page 42 OUTPUT BASED ANNUAL BUDGET\nHEA 9811 C...
42,43,OUTPUT BASED ANNUAL BUDGET Page 43\nHEA 9811 C...
43,44,Page 44 OUTPUT BASED ANNUAL BUDGET\nHEA 9811 C...


In [29]:
show_page(40)

Page 40 OUTPUT BASED ANNUAL BUDGET
HEA 9811 CHIRUNDU TOWN COUNCIL
D
BUDGET PROGRAMMES
Programme 16 : Agricultural Services
Programme Objective(S)
The Department of Agriculture Services has an operative objective of increasing Agriculture production and productivity.
This will be achieved through implementation of three (03) Sub-programs namely; namely Agriculture Crop production,
advisory and technical services, Agribusiness Development and Marketing and Management and Support Services with a
total allocation of K481, 034.00.
Table 4: Programme Budget Allocation by Economic Classification
2023 BUDGET 2024 BUDGET 2025 BUDGET
ECONOMIC CLASSIFICATION
Approved Expendit Approved Expendit
ure ure*
02 Use of Goods and Services 481,034
02 General Operations 481,034
The summary by economic classification shows that Agricultural Services programme has a total of
K481,034 which all go towards use of Goods and Services.



In [30]:
show_page(41)

OUTPUT BASED ANNUAL BUDGET Page 41
HEA 9811 CHIRUNDU TOWN COUNCIL
D
Programme 0016: Agricultural Services
Table 5: Programme Budget Allocation by Subprogramme
2023 BUDGET 2024 BUDGET
ure ure*
16 Agricultural Services (0) (0) 481,034
071 Agricultural Crop production, Advisory and | (0) | (0) | (0) | (0) | 281,034 |
Technical Services
072 Agribusiness Development and Marketing (0) (0) (0) (0) 65,000
073 Agriculture Co-ordination (0) (0) (0) (0) 135,000
The summary by economic classification shows that 58.4 percent (K 281,033) of the budget was allocated
to Agriculture Crop production, advisory and Technical Services including Agriculture Information
Services; 13.5 percent (K65,000) to Agribusiness Development and Marketing and 28.1 Percent (K135,00)
to Management and Support Services
The Agriculture Crop production, advisory and technical Services component with an allocation of 58.4
percent aims at increasing crop production and productivity through farm power and Mechanisation
promotio

In [31]:
show_page(42)

Page 42 OUTPUT BASED ANNUAL BUDGET
HEA 9811 CHIRUNDU TOWN COUNCIL
D
Programme: 16 Agricultural Services
Table 6: Programme Outputs
Key Output and Output Indicator 2023 2024 2025
Farm mechanization promoted
O01 Number of farmers trained in farm power and mechanization (0) (0) (0) 50 300
02 Number of Farmers accessing farm power and mechanization services (0) (0) (0) 100 300
03 Number of field officers trained in farm Power and Mecahnization (0) (0) (0) (0) 16
Irrigation Technologies promoted
01 Number of farmers trained in irrigation technologies (0) (0) (0) 200 500
02 Number of field officers trained in irrigation technologies (0) (0) (0) 16 16
Climate Smart Agricultural technologies promoted
01 Number of Farmers trained in Climate Smart Agriculture (0) (0) (0) 1,000 2,000
02 Number of farmer field school (FFS) functional (0) (0) (0) 35 52
03 Number of demonstration plots established (0) (0) (0) 13 13
04 Number of field officers trained in Climate Smart Agriculture (0) (0) (0) 16 16
Nu

In [32]:
programme16_data = [
    ("Farm mechanization promoted",
     "Number of farmers trained in farm power and mechanization", 300),

    ("Farm mechanization promoted",
     "Number of Farmers accessing farm power and mechanization services", 300),

    ("Farm mechanization promoted",
     "Number of field officers trained in farm Power and Mechanization", 16),

    ("Irrigation Technologies promoted",
     "Number of farmers trained in irrigation technologies", 500),

    ("Irrigation Technologies promoted",
     "Number of field officers trained in irrigation technologies", 16),

    ("Climate Smart Agricultural technologies promoted",
     "Number of Farmers trained in Climate Smart Agriculture", 2000),

    ("Climate Smart Agricultural technologies promoted",
     "Number of farmer field school (FFS) functional", 52),

    ("Climate Smart Agricultural technologies promoted",
     "Number of demonstration plots established", 13),

    ("Climate Smart Agricultural technologies promoted",
     "Number of field officers trained in Climate Smart Agriculture", 16),

    ("Nutrition education promoted among farmers",
     "Number of Farmers trained in nutrition", 1000),

    ("Nutrition education promoted among farmers",
     "Number of field officers trained in nutrition", 16),

    ("Crop diversification promoted",
     "Number of farmers trained in crop diversification", 3000),

    ("Crop diversification promoted",
     "Number of field officers trained in crop diversification", 16),

    ("Good farm management practices promoted",
     "Number of farmers trained in good farm management", 2000),

    ("Good farm management practices promoted",
     "Number of field officers trained in good farm management", 16),

    ("Extension service delivery enhanced",
     "Number of crop production demonstration plots established", 13),

    ("Extension service delivery enhanced",
     "Number of Farmer Field Schools operationalised", 52),

    ("Agriculture Information Services recorded and produced",
     "Number of agricultural radio programmes disseminated", 12),

    ("Agriculture Information Services recorded and produced",
     "Number of agricultural TV programmes disseminated", 4),

    ("Agriculture Information Services recorded and produced",
     "Number of agricultural publications disseminated", 4),

    ("Agriculture shows organised and exhibited",
     "Number of agricultural shows organised and exhibited", 5),

    ("Market information and bulletin developed and disseminated",
     "Number of commodity market bulletin produced", 52),

    ("Market information and bulletin developed and disseminated",
     "Number of commodity market bulletin disseminated", 52),

    ("Access to agricultural finance enhanced",
     "Number of farmers accessing agricultural finance", 500),

    ("Farmers trained in entrepreneurship",
     "Number of entrepreneurship trainings conducted", 4),

    ("Agricultural Trade facilitated",
     "Number of control of goods permits issued", 1000),

    ("Agricultural Trade facilitated",
     "Number of agricultural trade inspections conducted", 500)
]

programme16_rows = []

for key_output, indicator, target in programme16_data:
    programme16_rows.append({
        "year": YEAR,
        "category": "Output",
        "programme_or_output": indicator,
        "page": 42,
        "budget_amount": None,
        "currency": "ZMW",
        "notes": (
            "Programme 16: Agricultural Services; "
            f"Key output: {key_output}."
        ),
        "source_file": pdf_path.name,
        "target_value": target,
        "target_unit": "count"
    })

programme16_df = pd.DataFrame(
    programme16_rows,
    columns=extracted_columns
)

outputs_df = pd.concat(
    [outputs_df, programme16_df],
    ignore_index=True
)

print("Programme 16 rows added:", len(programme16_df))
print("Total output rows:", len(outputs_df))

Programme 16 rows added: 27
Total output rows: 166


In [33]:
outputs_df["programme_code"] = outputs_df["notes"].str.extract(
    r"Programme\s+(\d+)"
)[0]

print(
    outputs_df[outputs_df["programme_code"] == "16"].shape[0]
)

27


In [34]:
expected_programmes = {
    "1", "2", "3", "4", "5", "6", "7", "8",
    "10", "11", "12", "13", "15", "16", "17", "18"
}

found_programmes = set(outputs_df["programme_code"].dropna())

print("Missing programmes:", sorted(expected_programmes - found_programmes))
print("Output rows:", len(outputs_df))

Missing programmes: []
Output rows: 166


In [35]:
extracted_rows.extend(
    outputs_df[extracted_columns].to_dict("records")
)

final_df = pd.DataFrame(
    extracted_rows,
    columns=extracted_columns
)

final_df["budget_amount"] = pd.to_numeric(
    final_df["budget_amount"],
    errors="coerce"
).astype("Int64")

final_df["target_value"] = pd.to_numeric(
    final_df["target_value"],
    errors="coerce"
).astype("Int64")

final_file = output_folder / (
    "db-unza26-csc4792-"
    "chirundu_town_council_2025_extracted.csv"
)

final_df.to_csv(
    final_file,
    sep="|",
    index=False,
    encoding="utf-8-sig"
)

print(final_df["category"].value_counts())
print("Total final rows:", len(final_df))
print("Created:", final_file)

category
Output           166
Local Revenue     51
Programme         16
Revenue            4
CDF                1
LGEF               1
Name: count, dtype: int64
Total final rows: 239
Created: output_2025\db-unza26-csc4792-chirundu_town_council_2025_extracted.csv


In [36]:
check_df = pd.read_csv(final_file, sep="|")

print("Columns:", check_df.columns.tolist())
print("Rows read back:", len(check_df))

assert len(check_df) == 239
assert "category" in check_df.columns
assert "budget_amount" in check_df.columns
assert "target_value" in check_df.columns

print("2025 CSV validation passed.")

Columns: ['year', 'category', 'programme_or_output', 'page', 'budget_amount', 'currency', 'notes', 'source_file', 'target_value', 'target_unit']
Rows read back: 239
2025 CSV validation passed.
